<a href="https://colab.research.google.com/github/Sharmadipti/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sharmadipti/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use Logistic Regression as the first machine learning model for this project.

The target variable, went_dark, is binary:

1 = the page received zero clicks in the March outcome window
0 = the page continued receiving clicks
Because the task is a binary classification problem with a relatively small set of tabular features, Logistic Regression provides a simple and interpretable baseline model.

The model produces probabilities that can be used to rank pages by risk. This supports the project's goal of prioritizing pages for human review rather than making automatic decisions.

The Logistic Regression model will be compared against the Week-4 rule-based baseline using the same target and evaluation process.


In [3]:
import os
import getpass
import duckdb

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    return getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{get_hf_token()}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to the FlyRank warehouse.")

query_1 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(*) AS daily_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MAR}
GROUP BY client_hash_id, content_hash_id
ORDER BY daily_rows DESC
LIMIT 10
"""

grain_check = con.sql(query_1).df()

grain_check

query_2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MAR}
"""

slice_check = con.sql(query_2).df()

print("March 2026 slice:")
slice_check

query_3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc_data
FROM {MAR}
"""

availability_check = con.sql(query_3).df()

print("GSC data availability for March 2026:")
availability_check

# ---------------------------------------------------------
# Five-feature frame
# February = feature window
# March = label window
# ---------------------------------------------------------

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"

feature_query = f"""
WITH feb_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_feb,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_feb
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

march_label AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_mar
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_feb,
    f.clicks_feb,
    f.ctr_feb,
    f.avg_position_feb,
    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-02-28'
    ) AS content_age_days,
    CASE
        WHEN COALESCE(m.clicks_mar, 0) = 0 THEN 1
        ELSE 0
    END AS went_dark
FROM feb_features f
LEFT JOIN march_label m
    ON f.client_hash_id = m.client_hash_id
   AND f.content_hash_id = m.content_hash_id
LEFT JOIN {DIM} c
    ON f.content_hash_id = c.content_hash_id
WHERE f.impressions_feb >= 100
  AND f.clicks_feb >= 3
  AND c.is_published IS TRUE
  AND c.content_created_date <= DATE '2026-02-28'
"""

feature_frame = con.sql(feature_query).df()

print(f"Feature-frame rows: {len(feature_frame):,}")
print("\nFive features:")
print([
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
])

feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "content_age_days",
        "went_dark"
    ]
].head()



Paste your Hugging Face READ token: ··········
Connected to the FlyRank warehouse.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 slice:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GSC data availability for March 2026:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 29,700

Five features:
['impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'content_age_days']


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,content_age_days,went_dark
0,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,3072.0,6.0,0.001953,8.544596,16,0
1,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,1047.0,8.0,0.007641,2.753582,16,0
2,client_62f4a7e64f5e0096,content_bb843e565f31bb7b,1195.0,6.0,0.005021,1.676987,16,0
3,client_62f4a7e64f5e0096,content_488fdbdd80d70a88,1813.0,5.0,0.002758,3.642581,10,0
4,client_9958f0a7ae1df715,content_55ead56c1217a888,218.0,3.0,0.013761,8.678899,337,0


In [4]:
print("Feature frame shape:", feature_frame.shape)

print("\nTarget distribution:")
print(feature_frame["went_dark"].value_counts())

print("\nTarget distribution (%):")
print(
    feature_frame["went_dark"]
    .value_counts(normalize=True)
    .round(3)
)

Feature frame shape: (29700, 8)

Target distribution:
went_dark
0    28194
1     1506
Name: count, dtype: int64

Target distribution (%):
went_dark
0    0.949
1    0.051
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The feature frame contains a single February 2026 feature window and a March 2026 outcome window. Because only one feature-to-outcome snapshot is available in this assignment, a true time-based split is not possible.

To evaluate the model honestly, I will use a stratified train/test split. Stratification helps preserve the proportion of pages that went dark and pages that remained active in both datasets.

The same split will be used for both the machine learning model and the Week-4 rule-based baseline so that the comparison is fair.

This evaluation should be interpreted as a holdout test on the available snapshot rather than a future-period validation.

In [5]:
from sklearn.model_selection import train_test_split

feature_cols = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
]

X = feature_frame[feature_cols].fillna(
    feature_frame[feature_cols].median()
)

y = feature_frame["went_dark"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train rows:", len(X_train))
print("Test rows :", len(X_test))

print("\nTrain distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train rows: 22275
Test rows : 7425

Train distribution:
went_dark
0    0.94927
1    0.05073
Name: proportion, dtype: float64

Test distribution:
went_dark
0    0.94936
1    0.05064
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
A Logistic Regression model was trained using the same five features developed in the data contract stage:

impressions_feb
clicks_feb
ctr_feb
avg_position_feb
content_age_days
Because the target class is highly imbalanced, a balanced Logistic Regression model was evaluated. The goal is not to automate refresh decisions but to prioritize pages for human review.

The model performance will be compared with the Week-4 rule-based baseline using the same target definition.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Train model
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.5972

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.58      0.73      7049
           1       0.10      0.84      0.17       376

    accuracy                           0.60      7425
   macro avg       0.54      0.71      0.45      7425
weighted avg       0.94      0.60      0.71      7425



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
The initial Logistic Regression model achieved high accuracy but failed to identify any pages in the minority went_dark class. This occurred because the dataset is highly imbalanced, with only about 5% of pages belonging to the positive class.

To address this issue, a balanced Logistic Regression model was trained using class_weight="balanced". This reduced overall accuracy but substantially increased recall for pages that went dark.

The balanced model identified most of the pages that went dark (recall = 0.84), but it also generated many false positives. This behavior may be acceptable in a decision-support workflow because missing a page that requires review can be more costly than reviewing an extra page.

The results should be interpreted as directional and supportive of prioritization rather than as an automated decision system.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.